# 🛒 Olist ETL Pipeline

**Full pipeline — Extract → Transform → Load → Validate → Report → Cleanup**

This notebook replaces the Apache Airflow DAG (`olist_etl_dag.py`) and runs the entire ETL pipeline sequentially in one place.

| Stage | Description |
|-------|-------------|
| **1. Extract** | Load Olist CSVs, fetch São Paulo Weather API, convert Orders to Parquet |
| **2. Transform** | PySpark joins + feature engineering (falls back to pandas if Java absent) |
| **3. Load** | Persist transformed Parquet files into DuckDB analytical tables |
| **4. Validate** | Row-count + NULL checks on every DuckDB table |
| **5. Report** | Write `pipeline_report.json` with KPIs, top states, monthly trend |
| **6. Cleanup** | Remove Spark temp dirs and notebook output artefacts |

---
> **Run order:** Execute cells top-to-bottom, or use **Run All**.

## ⚙️ Global Setup

In [ ]:
import os, json, glob, shutil, subprocess, sys
from datetime import datetime

# ── Project paths ──────────────────────────────────────────
PROJECT_ROOT = os.path.abspath('..')          # one level above notebooks/
RAW          = os.path.join(PROJECT_ROOT, 'data', 'raw')
PARQUET      = os.path.join(PROJECT_ROOT, 'data', 'parquet')
TRANSFORMED  = os.path.join(PROJECT_ROOT, 'data', 'transformed')
DB_PATH      = os.path.join(PROJECT_ROOT, 'analytics.duckdb')
REPORT_PATH  = os.path.join(PROJECT_ROOT, 'pipeline_report.json')

for folder in [RAW, PARQUET, TRANSFORMED]:
    os.makedirs(folder, exist_ok=True)

print('Project root :', PROJECT_ROOT)
print('RAW          :', RAW)
print('PARQUET      :', PARQUET)
print('TRANSFORMED  :', TRANSFORMED)
print('DB_PATH      :', DB_PATH)
print('Folders ready ✅')

---
## Stage 1 — Extract

Pulls data from three sources:
- **Source 1** – Olist CSV files (local `data/raw/`)
- **Source 2** – São Paulo Weather via Open-Meteo REST API
- **Source 3** – Convert Olist Orders CSV → Parquet

In [ ]:
import pandas as pd
import requests

print('=' * 60)
print('STAGE 1 / 6 — EXTRACT')
print('=' * 60)

In [ ]:
# ── Source 1 : Load Olist CSV files ───────────────────────
orders      = pd.read_csv(f'{RAW}/olist_orders_dataset.csv')
customers   = pd.read_csv(f'{RAW}/olist_customers_dataset.csv')
products    = pd.read_csv(f'{RAW}/olist_products_dataset.csv')
order_items = pd.read_csv(f'{RAW}/olist_order_items_dataset.csv')
payments    = pd.read_csv(f'{RAW}/olist_order_payments_dataset.csv')
reviews     = pd.read_csv(f'{RAW}/olist_order_reviews_dataset.csv')
sellers     = pd.read_csv(f'{RAW}/olist_sellers_dataset.csv')
geolocation = pd.read_csv(f'{RAW}/olist_geolocation_dataset.csv')
category    = pd.read_csv(f'{RAW}/product_category_name_translation.csv')

print('Source 1 — Olist CSV files loaded')
print(f'  orders     : {orders.shape}')
print(f'  customers  : {customers.shape}')
print(f'  products   : {products.shape}')
print(f'  order_items: {order_items.shape}')
print(f'  payments   : {payments.shape}')
print(f'  reviews    : {reviews.shape}')
orders.head()

In [ ]:
# ── Source 2 : São Paulo Weather API (Open-Meteo) ─────────
url = 'https://archive-api.open-meteo.com/v1/archive'
params = {
    'latitude'  : -23.5505,
    'longitude' : -46.6333,
    'start_date': '2017-01-01',
    'end_date'  : '2018-08-31',
    'daily'     : 'temperature_2m_max,precipitation_sum',
    'timezone'  : 'America/Sao_Paulo'
}

response = requests.get(url, params=params)
print(f'API Status: {response.status_code}')

data    = response.json()
weather = pd.DataFrame({
    'date'         : data['daily']['time'],
    'temp_max'     : data['daily']['temperature_2m_max'],
    'precipitation': data['daily']['precipitation_sum']
})
weather['date'] = pd.to_datetime(weather['date'])
weather_csv_path = f'{RAW}/sao_paulo_weather.csv'
weather.to_csv(weather_csv_path, index=False)
print(f'Source 2 — Weather data saved: {len(weather):,} rows → {weather_csv_path}')
weather.head()

In [ ]:
# ── Source 3 : Convert Orders CSV → Parquet ───────────────
orders['order_purchase_timestamp']        = pd.to_datetime(orders['order_purchase_timestamp'])
orders['order_delivered_customer_date']   = pd.to_datetime(orders['order_delivered_customer_date'])
orders['order_estimated_delivery_date']   = pd.to_datetime(orders['order_estimated_delivery_date'])

parquet_path = f'{PARQUET}/olist_orders.parquet'
orders.to_parquet(parquet_path, index=False)

verify = pd.read_parquet(parquet_path)
print(f'Source 3 — Parquet saved: {parquet_path}')
print(f'  Shape : {verify.shape}')

print('\n' + '=' * 60)
print('✅  STAGE 1 COMPLETE — Extraction done')
print(f'  Source 1 – Olist CSVs   : {orders.shape[0]:,} orders')
print(f'  Source 2 – Weather API  : {len(weather):,} days')
print(f'  Source 3 – Parquet file : olist_orders.parquet created')
print('=' * 60)

---
## Stage 2 — Transform (PySpark / pandas fallback)

Applies business-logic transformations:
- Joins orders, customers, payments, items, and weather data
- Engineers `delivery_days`, `was_late`, weather buckets
- Writes four Parquet tables to `data/transformed/`

> If Java / PySpark is unavailable the cell automatically falls back to a pure-pandas implementation.

In [ ]:
print('=' * 60)
print('STAGE 2 / 6 — TRANSFORM')
print('=' * 60)

# Detect Java availability
java_available = False
try:
    check = subprocess.run(['java', '-version'], capture_output=True, timeout=10)
    java_available = (check.returncode == 0)
except (FileNotFoundError, subprocess.TimeoutExpired):
    pass

print(f'Java detected: {java_available}')
print('Will use: PySpark' if java_available else 'Will use: pandas (fallback)')

In [ ]:
if java_available:
    # ── PySpark path ───────────────────────────────────────
    import os
    # Adjust these paths to match your local Java / Spark install
    # os.environ['JAVA_HOME']  = r'C:\Program Files\Java\jdk-17'   # Windows example
    # os.environ['SPARK_HOME'] = r'C:\spark'
    # os.environ['HADOOP_HOME']= r'C:\hadoop'

    from pyspark.sql import SparkSession
    from pyspark.sql.functions import (
        col, to_date, avg, count,
        sum as spark_sum, round as spark_round,
        datediff, when, month, year
    )

    spark = (
        SparkSession.builder
        .appName('OlistETL')
        .master('local[*]')
        .config('spark.driver.memory', '2g')
        .config('spark.sql.shuffle.partitions', '4')
        .config('spark.driver.host', '127.0.0.1')
        .config('spark.driver.bindAddress', '127.0.0.1')
        .config('spark.sql.parquet.outputTimestampType', 'TIMESTAMP_MICROS')
        .config('spark.sql.parquet.int96RebaseModeInRead',    'CORRECTED')
        .config('spark.sql.parquet.int96RebaseModeInWrite',   'CORRECTED')
        .config('spark.sql.parquet.datetimeRebaseModeInRead', 'CORRECTED')
        .config('spark.sql.parquet.datetimeRebaseModeInWrite','CORRECTED')
        .getOrCreate()
    )
    spark.sparkContext.setLogLevel('ERROR')
    print('Spark version:', spark.version, '— session started ✅')

    # Load CSVs into Spark
    sp_orders      = spark.read.csv(f'{RAW}/olist_orders_dataset.csv',          header=True, inferSchema=True)
    sp_customers   = spark.read.csv(f'{RAW}/olist_customers_dataset.csv',       header=True, inferSchema=True)
    sp_order_items = spark.read.csv(f'{RAW}/olist_order_items_dataset.csv',     header=True, inferSchema=True)
    sp_payments    = spark.read.csv(f'{RAW}/olist_order_payments_dataset.csv',  header=True, inferSchema=True)
    sp_weather     = spark.read.csv(f'{RAW}/sao_paulo_weather.csv',             header=True, inferSchema=True)

    print('All datasets loaded into Spark.')

    # Transformation 1 — Clean orders
    orders_clean = (
        sp_orders
        .filter(col('order_status') == 'delivered')
        .withColumn('purchase_date',  to_date(col('order_purchase_timestamp')))
        .withColumn('delivery_date',  to_date(col('order_delivered_customer_date')))
        .withColumn('estimated_date', to_date(col('order_estimated_delivery_date')))
        .withColumn('delivery_days',  datediff(col('delivery_date'), col('purchase_date')))
        .withColumn('was_late',       when(col('delivery_date') > col('estimated_date'), 1).otherwise(0))
        .withColumn('year',           year(col('purchase_date')))
        .withColumn('month',          month(col('purchase_date')))
    )
    print('Delivered orders:', orders_clean.count())

    # Transformation 2 — Join customers, payments, items
    oc = orders_clean.join(
        sp_customers.select('customer_id', 'customer_state', 'customer_city'),
        on='customer_id', how='left'
    )
    pay_agg = sp_payments.groupBy('order_id').agg(
        spark_sum('payment_value').alias('total_payment'),
        count('payment_sequential').alias('payment_count')
    )
    items_agg = sp_order_items.groupBy('order_id').agg(
        count('order_item_id').alias('item_count'),
        spark_sum('price').alias('items_total'),
        spark_sum('freight_value').alias('freight_total')
    )
    orders_full_sp = oc.join(pay_agg, on='order_id', how='left').join(items_agg, on='order_id', how='left')

    # Transformation 3 — Join weather
    weather_clean = (
        sp_weather
        .withColumn('date',          to_date(col('date')))
        .withColumn('temp_max',      col('temp_max').cast('double'))
        .withColumn('precipitation', col('precipitation').cast('double'))
    )
    orders_weather_sp = orders_full_sp.join(
        weather_clean,
        orders_full_sp['purchase_date'] == weather_clean['date'],
        how='left'
    ).drop('date')

    # Transformation 4 — Aggregated summary tables
    monthly_revenue_sp = orders_weather_sp.groupBy('year', 'month').agg(
        count('order_id').alias('total_orders'),
        spark_round(spark_sum('total_payment'), 2).alias('total_revenue'),
        spark_round(avg('total_payment'), 2).alias('avg_order_value'),
        spark_round(avg('delivery_days'), 1).alias('avg_delivery_days'),
        spark_sum('was_late').alias('late_deliveries')
    ).orderBy('year', 'month')

    revenue_by_state_sp = orders_weather_sp.groupBy('customer_state').agg(
        count('order_id').alias('total_orders'),
        spark_round(spark_sum('total_payment'), 2).alias('total_revenue'),
        spark_round(avg('total_payment'), 2).alias('avg_order_value'),
        spark_round(avg('delivery_days'), 1).alias('avg_delivery_days')
    ).orderBy('total_revenue', ascending=False)

    weather_impact_sp = (
        orders_weather_sp
        .groupBy('precipitation').agg(
            count('order_id').alias('total_orders'),
            spark_round(avg('total_payment'), 2).alias('avg_order_value')
        )
        .withColumn('rain_category',
            when(col('precipitation') == 0, 'No Rain')
            .when(col('precipitation') < 5,  'Light Rain')
            .otherwise('Heavy Rain')
        )
        .groupBy('rain_category').agg(
            spark_sum('total_orders').alias('total_orders'),
            spark_round(avg('avg_order_value'), 2).alias('avg_order_value')
        )
    )

    # Save to Parquet
    orders_weather_sp.write.mode('overwrite').parquet(f'{TRANSFORMED}/orders_full')
    monthly_revenue_sp.write.mode('overwrite').parquet(f'{TRANSFORMED}/monthly_revenue')
    revenue_by_state_sp.write.mode('overwrite').parquet(f'{TRANSFORMED}/revenue_by_state')
    weather_impact_sp.write.mode('overwrite').parquet(f'{TRANSFORMED}/weather_impact')

    spark.stop()
    print('Spark session stopped.')
    print('\n✅  STAGE 2 COMPLETE — PySpark transform done')

else:
    # ── pandas fallback path ───────────────────────────────
    print('Running pandas fallback transform...')
    import numpy as np

    # Reload with parsed dates (orders may already have them from Stage 1)
    ord_df  = pd.read_csv(f'{RAW}/olist_orders_dataset.csv',         parse_dates=['order_purchase_timestamp','order_delivered_customer_date','order_estimated_delivery_date'])
    cust_df = pd.read_csv(f'{RAW}/olist_customers_dataset.csv')
    pay_df  = pd.read_csv(f'{RAW}/olist_order_payments_dataset.csv')
    itm_df  = pd.read_csv(f'{RAW}/olist_order_items_dataset.csv')
    wthr_df = pd.read_csv(f'{RAW}/sao_paulo_weather.csv',            parse_dates=['date'])

    # Clean orders — delivered only
    ord_df = ord_df[ord_df['order_status'] == 'delivered'].copy()
    ord_df['purchase_date']  = ord_df['order_purchase_timestamp'].dt.normalize()
    ord_df['delivery_date']  = ord_df['order_delivered_customer_date'].dt.normalize()
    ord_df['estimated_date'] = ord_df['order_estimated_delivery_date'].dt.normalize()
    ord_df['delivery_days']  = (ord_df['delivery_date'] - ord_df['purchase_date']).dt.days
    ord_df['was_late']       = (ord_df['delivery_date'] > ord_df['estimated_date']).astype(int)
    ord_df['year']           = ord_df['purchase_date'].dt.year
    ord_df['month']          = ord_df['purchase_date'].dt.month

    # Aggregations
    pay_agg  = pay_df.groupby('order_id').agg(total_payment=('payment_value','sum'), payment_count=('payment_sequential','count')).reset_index()
    item_agg = itm_df.groupby('order_id').agg(item_count=('order_item_id','count'), items_total=('price','sum'), freight_total=('freight_value','sum')).reset_index()

    # Joins
    df = ord_df.merge(cust_df[['customer_id','customer_state','customer_city']], on='customer_id', how='left')
    df = df.merge(pay_agg,  on='order_id', how='left')
    df = df.merge(item_agg, on='order_id', how='left')
    df = df.merge(wthr_df,  left_on='purchase_date', right_on='date', how='left').drop(columns=['date'], errors='ignore')

    # Summary tables
    monthly_revenue_pd = df.groupby(['year','month']).agg(
        total_orders=('order_id','count'),
        total_revenue=('total_payment','sum'),
        avg_order_value=('total_payment','mean'),
        avg_delivery_days=('delivery_days','mean'),
        late_deliveries=('was_late','sum')
    ).round(2).reset_index().sort_values(['year','month'])

    revenue_by_state_pd = df.groupby('customer_state').agg(
        total_orders=('order_id','count'),
        total_revenue=('total_payment','sum'),
        avg_order_value=('total_payment','mean'),
        avg_delivery_days=('delivery_days','mean')
    ).round(2).reset_index().sort_values('total_revenue', ascending=False)

    def rain_cat(p):
        if p == 0: return 'No Rain'
        if p < 5:  return 'Light Rain'
        return 'Heavy Rain'

    df['rain_category'] = df['precipitation'].fillna(0).apply(rain_cat)
    weather_impact_pd = df.groupby('rain_category').agg(
        total_orders=('order_id','count'),
        avg_order_value=('total_payment','mean')
    ).round(2).reset_index()

    # Save to Parquet
    os.makedirs(f'{TRANSFORMED}/orders_full',       exist_ok=True)
    os.makedirs(f'{TRANSFORMED}/monthly_revenue',   exist_ok=True)
    os.makedirs(f'{TRANSFORMED}/revenue_by_state',  exist_ok=True)
    os.makedirs(f'{TRANSFORMED}/weather_impact',    exist_ok=True)

    df.to_parquet(                f'{TRANSFORMED}/orders_full/part-0.parquet',      index=False)
    monthly_revenue_pd.to_parquet(f'{TRANSFORMED}/monthly_revenue/part-0.parquet',  index=False)
    revenue_by_state_pd.to_parquet(f'{TRANSFORMED}/revenue_by_state/part-0.parquet',index=False)
    weather_impact_pd.to_parquet( f'{TRANSFORMED}/weather_impact/part-0.parquet',   index=False)

    print('\n✅  STAGE 2 COMPLETE — pandas fallback transform done')
    print(f'  orders_full      : {len(df):,} rows')
    print(f'  monthly_revenue  : {len(monthly_revenue_pd):,} rows')
    print(f'  revenue_by_state : {len(revenue_by_state_pd):,} rows')
    print(f'  weather_impact   : {len(weather_impact_pd):,} rows')

---
## Stage 3 — Load into DuckDB

Reads the four Parquet tables from `data/transformed/` and loads them into **analytics.duckdb** as analytical tables.

In [ ]:
import duckdb

print('=' * 60)
print('STAGE 3 / 6 — LOAD')
print('=' * 60)

conn = duckdb.connect(DB_PATH)
print('Connected to DuckDB:', DB_PATH)

tables_to_load = {
    'orders_full'      : f'{TRANSFORMED}/orders_full/*.parquet',
    'monthly_revenue'  : f'{TRANSFORMED}/monthly_revenue/*.parquet',
    'revenue_by_state' : f'{TRANSFORMED}/revenue_by_state/*.parquet',
    'weather_impact'   : f'{TRANSFORMED}/weather_impact/*.parquet',
}

for table, pattern in tables_to_load.items():
    order_clause = 'ORDER BY year, month' if table == 'monthly_revenue' else \
                   'ORDER BY total_revenue DESC' if table == 'revenue_by_state' else ''
    conn.execute(f"""
        CREATE OR REPLACE TABLE {table} AS
        SELECT * FROM read_parquet('{pattern}')
        {order_clause}
    """)
    count = conn.execute(f'SELECT COUNT(*) FROM {table}').fetchone()[0]
    print(f'  {table:<25}: {count:,} rows loaded ✅')

print('\nTables in DuckDB:')
print(conn.execute('SHOW TABLES').fetchdf())

print('\n✅  STAGE 3 COMPLETE — Load done')

In [ ]:
# Quick preview — monthly revenue
conn.execute('SELECT * FROM monthly_revenue LIMIT 6').fetchdf()

In [ ]:
# Top 5 states by revenue
conn.execute('SELECT * FROM revenue_by_state LIMIT 5').fetchdf()

In [ ]:
# Weather impact
conn.execute('SELECT * FROM weather_impact').fetchdf()

---
## Stage 4 — Data Quality Validation

- **Row-count check** – every table must have > 0 rows
- **NULL check** – `order_id`, `total_payment`, `customer_state` must be non-null

In [ ]:
print('=' * 60)
print('STAGE 4 / 6 — VALIDATE')
print('=' * 60)

tables   = ['orders_full', 'monthly_revenue', 'revenue_by_state', 'weather_impact']
failures = []

# Row-count check
print('Row-count checks:')
for table in tables:
    try:
        count = conn.execute(f'SELECT COUNT(*) FROM {table}').fetchone()[0]
        if count == 0:
            failures.append(f'{table}: 0 rows (expected > 0)')
            print(f'  ❌  {table}: 0 rows')
        else:
            print(f'  ✅  {table}: {count:,} rows')
    except Exception as exc:
        failures.append(f'{table}: query error — {exc}')
        print(f'  ❌  {table}: {exc}')

# NULL check on critical columns
print('\nNULL check — orders_full critical columns:')
null_df = conn.execute('''
    SELECT
        SUM(CASE WHEN order_id       IS NULL THEN 1 ELSE 0 END) AS null_order_ids,
        SUM(CASE WHEN total_payment  IS NULL THEN 1 ELSE 0 END) AS null_payments,
        SUM(CASE WHEN customer_state IS NULL THEN 1 ELSE 0 END) AS null_states
    FROM orders_full
''').fetchdf()
print(null_df.to_string(index=False))

for col_name in ['null_order_ids', 'null_payments', 'null_states']:
    val = int(null_df[col_name].iloc[0])
    if val > 0:
        failures.append(f'orders_full.{col_name.replace("null_","")}: {val} NULL values')

if failures:
    print('\n❌  Data quality failures:')
    for f in failures:
        print(f'  • {f}')
    raise ValueError('Data quality validation failed — see above.')
else:
    print('\n✅  STAGE 4 COMPLETE — All quality checks passed')

---
## Stage 5 — Generate Pipeline Report

Queries DuckDB and writes `pipeline_report.json` with key KPIs, top states, and monthly trend.

In [ ]:
print('=' * 60)
print('STAGE 5 / 6 — REPORT')
print('=' * 60)

kpis = conn.execute('''
    SELECT
        COUNT(order_id)                                       AS total_orders,
        ROUND(SUM(total_payment), 2)                          AS total_revenue,
        ROUND(AVG(total_payment), 2)                          AS avg_order_value,
        ROUND(AVG(delivery_days), 1)                          AS avg_delivery_days,
        SUM(was_late)                                         AS total_late_deliveries,
        ROUND(SUM(was_late) * 100.0 / COUNT(order_id), 2)    AS late_delivery_rate_pct
    FROM orders_full
''').fetchdf().to_dict('records')[0]

top_states = conn.execute('''
    SELECT customer_state, total_orders, total_revenue
    FROM   revenue_by_state
    ORDER  BY total_revenue DESC
    LIMIT  5
''').fetchdf().to_dict('records')

monthly_trend = conn.execute('''
    SELECT year, month, total_orders, total_revenue
    FROM   monthly_revenue
    ORDER  BY year, month
    LIMIT  6
''').fetchdf().to_dict('records')

report = {
    'pipeline_run_time': datetime.now().isoformat(),
    'key_metrics'      : kpis,
    'top_states'       : top_states,
    'monthly_trend'    : monthly_trend,
}

with open(REPORT_PATH, 'w') as fh:
    json.dump(report, fh, indent=2, default=str)

conn.close()

print(f'Report saved → {REPORT_PATH}')
print(f"  Total Orders       : {kpis['total_orders']:,}")
print(f"  Total Revenue      : R${kpis['total_revenue']:,.2f}")
print(f"  Avg Order Value    : R${kpis['avg_order_value']:,.2f}")
print(f"  Avg Delivery Days  : {kpis['avg_delivery_days']}")
print(f"  Late Delivery Rate : {kpis['late_delivery_rate_pct']}%")
print('\n✅  STAGE 5 COMPLETE — Report generated')

### Report preview

In [ ]:
with open(REPORT_PATH) as fh:
    print(json.dumps(json.load(fh), indent=2))

---
## Stage 6 — Cleanup

Removes Spark temp directories (`spark-warehouse/`, `metastore_db/`, `derby.log`) and any `*_output.ipynb` / `*.pyc` files.

In [ ]:
print('=' * 60)
print('STAGE 6 / 6 — CLEANUP')
print('=' * 60)

notebooks_dir = os.path.abspath('.')   # current notebook dir

# Notebook output files and bytecode
for pattern in ['*_output.ipynb', '*.pyc']:
    for f in glob.glob(os.path.join(notebooks_dir, pattern)):
        os.remove(f)
        print(f'  Removed file : {f}')

# Spark / Derby artefacts
for artefact in ['spark-warehouse', 'metastore_db', 'derby.log']:
    path = os.path.join(PROJECT_ROOT, artefact)
    if os.path.isdir(path):
        shutil.rmtree(path)
        print(f'  Removed dir  : {path}')
    elif os.path.isfile(path):
        os.remove(path)
        print(f'  Removed file : {path}')

print('\n✅  STAGE 6 COMPLETE — Cleanup done')

---
## Pipeline Summary

In [ ]:
print('=' * 60)
print('PIPELINE COMPLETE ✅')
print('=' * 60)

stages = [
    ('1 Extract',   'CSVs + Weather API + Parquet'),
    ('2 Transform', 'PySpark / pandas — 4 Parquet tables'),
    ('3 Load',      'DuckDB — 4 analytical tables'),
    ('4 Validate',  'Row-count + NULL checks'),
    ('5 Report',    f'pipeline_report.json → {REPORT_PATH}'),
    ('6 Cleanup',   'Spark temp dirs + output notebooks'),
]

for stage, desc in stages:
    print(f'  ✅  Stage {stage:<12}  {desc}')

print('=' * 60)
print(f'DuckDB  : {DB_PATH}')
print(f'Report  : {REPORT_PATH}')
print(f'Run time: {datetime.now().isoformat()}')
print('=' * 60)